# Second-Order Methods — Theory Notebook

> *"If gradient descent is walking downhill with your eyes closed, Newton's method is looking at the entire terrain."*

Interactive demonstrations of Newton's method, BFGS, Gauss-Newton, natural gradient, and K-FAC. Companion to [notes.md](notes.md).

In [ ]:
import numpy as np
import scipy.linalg as la
import scipy.optimize as opt

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
        HAS_SNS = True
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
        HAS_SNS = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {
    'primary': '#0077BB',
    'secondary': '#EE7733',
    'tertiary': '#009988',
    'error': '#CC3311',
    'neutral': '#555555',
    'highlight': '#EE3377',
}

if HAS_MPL:
    mpl.rcParams.update({
        'figure.figsize': (10, 6),
        'figure.dpi': 120,
        'font.size': 13,
        'axes.titlesize': 15,
        'axes.labelsize': 13,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'lines.linewidth': 2.0,
        'axes.spines.top': False,
        'axes.spines.right': False,
    })

np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)
print('Setup complete.')

---

## 1. Newton's Method: Quadratic Convergence

Demonstrate Newton's method on a non-quadratic function and verify quadratic convergence.

In [ ]:
# === 1.1 Newton's method on f(x) = x^4 - 4x^3 + 2x^2 + 4x ===
def f(x):
    return x**4 - 4*x**3 + 2*x**2 + 4*x

def grad_f(x):
    return 4*x**3 - 12*x**2 + 4*x + 4

def hess_f(x):
    return 12*x**2 - 24*x + 4

# Roots of f'(x) = 0: x ≈ -0.414, 1.0, 2.414
# f''(1) = 12 - 24 + 4 = -8 < 0 (local max)
# f''(-0.414) > 0, f''(2.414) > 0 (local minima)

x = 2.5  # Start near the minimum at x ≈ 2.414
T = 10
x_history = [x]
err_history = [abs(x - 2.41421356)]

for t in range(T):
    g = grad_f(x)
    H = hess_f(x)
    if abs(H) < 1e-10:
        print(f'Singular Hessian at iteration {t}')
        break
    d = -g / H
    x = x + d
    x_history.append(x)
    err_history.append(abs(x - 2.41421356))

print(f'True minimum: x* ≈ 2.41421356 (1 + sqrt(2))')
print(f'Newton iterates:')
for i, (x_val, err) in enumerate(zip(x_history, err_history)):
    print(f'  t={i}: x = {x_val:.10f}, error = {err:.2e}')

# Verify quadratic convergence
if len(err_history) >= 4:
    for i in range(2, min(5, len(err_history))):
        if err_history[i-1] > 1e-15:
            ratio = err_history[i] / err_history[i-1]**2
            print(f'  err[{i}] / err[{i-1}]^2 = {ratio:.4f} (should be constant)')

ok = err_history[-1] < 1e-12
print(f"\n{'PASS' if ok else 'FAIL'} — Newton converged to machine precision")

In [ ]:
# === 1.2 Plot convergence ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Function plot
    x_plot = np.linspace(-1, 4, 200)
    axes[0].plot(x_plot, f(x_plot), color=COLORS['primary'])
    axes[0].plot(x_history[:len(err_history)], [f(x) for x in x_history[:len(err_history)]],
                 'ro-', markersize=6, label='Newton iterates')
    axes[0].axhline(0, color='black', alpha=0.1)
    axes[0].set_title('Newton\'s method on f(x)')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].legend()
    
    # Error plot
    steps = np.arange(len(err_history))
    axes[1].plot(steps, err_history, 'o-', color=COLORS['error'])
    axes[1].set_yscale('log')
    axes[1].set_title('Quadratic convergence: error vs iteration')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('|x_t - x*| (log scale)')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 2. Newton's Method vs. Gradient Descent

Compare convergence rates on an ill-conditioned quadratic.

In [ ]:
# === 2.1 Newton vs GD on 2D quadratic ===
kappa = 100
A = np.diag([1.0, kappa])
L = kappa
x0 = np.array([1.0, 1.0])
T = 50

# GD
x = x0.copy()
err_gd = []
for _ in range(T):
    x = x - (1/L) * (A @ x)
    err_gd.append(0.5 * x @ A @ x)

# Newton
x = x0.copy()
err_newton = []
for _ in range(T):
    x = x - np.linalg.solve(A, A @ x)  # A^{-1} A x = x, so x - x = 0
    err_newton.append(0.5 * x @ A @ x)

err_gd = np.array(err_gd)
err_newton = np.array(err_newton)

print(f'Condition number: kappa = {kappa}')
print(f'GD after {T} steps: error = {err_gd[-1]:.2e}')
print(f'Newton after 1 step: error = {err_newton[0]:.2e}')
print(f'Newton after 2 steps: error = {err_newton[1]:.2e}')

ok = err_newton[0] < 1e-20
print(f"\n{'PASS' if ok else 'FAIL'} — Newton converges in one step for quadratics")

In [ ]:
# === 2.2 Plot comparison ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    
    ax.plot(steps, err_gd, color=COLORS['neutral'], label='GD')
    ax.plot(steps[:2], err_newton[:2], 'o-', color=COLORS['primary'],
            markersize=8, label='Newton')
    
    ax.set_yscale('log')
    ax.set_title('Newton vs GD on ill-conditioned quadratic')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Suboptimality')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 3. Damped Newton's Method with Backtracking

Globalize Newton's method using Armijo backtracking.

In [ ]:
# === 3.1 Damped Newton on Rosenbrock function ===
def rosenbrock(x):
    return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

def rosenbrock_grad(x):
    dx = -2*(1-x[0]) - 400*x[0]*(x[1]-x[0]**2)
    dy = 200*(x[1]-x[0]**2)
    return np.array([dx, dy])

def rosenbrock_hess(x):
    H00 = 2 - 400*x[1] + 1200*x[0]**2
    H01 = -400*x[0]
    H11 = 200
    return np.array([[H00, H01], [H01, H11]])

def damped_newton(f, grad_f, hess_f, x0, T=100, alpha=0.01, beta=0.5):
    x = x0.copy().astype(float)
    x_history = [x.copy()]
    f_history = [f(x)]
    eta_history = []
    
    for t in range(T):
        g = grad_f(x)
        H = hess_f(x)
        
        # Regularize if needed
        eigvals = la.eigvalsh(H)
        if eigvals[0] < 1e-8:
            H = H + (1e-8 - eigvals[0]) * np.eye(len(x))
        
        d = -la.solve(H, g)
        
        # Backtracking line search
        eta = 1.0
        f_x = f(x)
        slope = g @ d
        while f(x + eta*d) > f_x + alpha*eta*slope:
            eta *= beta
            if eta < 1e-16:
                break
        eta_history.append(eta)
        
        x = x + eta * d
        x_history.append(x.copy())
        f_history.append(f(x))
        
        if np.linalg.norm(g) < 1e-10:
            break
    
    return np.array(x_history), np.array(f_history), np.array(eta_history)

x0 = np.array([0.0, 0.0])
x_hist, f_hist, eta_hist = damped_newton(rosenbrock, rosenbrock_grad, rosenbrock_hess, x0, T=50)

print(f'Rosenbrock: f*(1,1) = 0')
print(f'Start: {x0}, f = {f_hist[0]:.4f}')
print(f'Final: ({x_hist[-1][0]:.6f}, {x_hist[-1][1]:.6f}), f = {f_hist[-1]:.2e}')
print(f'Iterations: {len(f_hist)-1}')
print(f'Average step size: {np.mean(eta_hist):.4f}')

ok = f_hist[-1] < 1e-8
print(f"\n{'PASS' if ok else 'FAIL'} — damped Newton converged")

---

## 4. BFGS Quasi-Newton Method

Implement and analyze the BFGS update formula.

In [ ]:
# === 4.1 BFGS implementation ===
def bfgs(f, grad_f, x0, T=100, tol=1e-10):
    n = len(x0)
    x = x0.copy().astype(float)
    H = np.eye(n)  # Initial inverse Hessian approximation
    x_history = [x.copy()]
    f_history = [f(x)]
    
    for t in range(T):
        g = grad_f(x)
        if np.linalg.norm(g) < tol:
            break
        
        # Search direction
        d = -H @ g
        
        # Line search (backtracking)
        eta = 1.0
        f_x = f(x)
        slope = g @ d
        while f(x + eta*d) > f_x + 0.01*eta*slope:
            eta *= 0.5
        
        s = eta * d
        x_new = x + s
        g_new = grad_f(x_new)
        y = g_new - g
        
        # BFGS update
        ys = y @ s
        if ys > 1e-10:  # Only update if curvature condition holds
            rho = 1.0 / ys
            I = np.eye(n)
            V = I - rho * np.outer(s, y)
            H = V @ H @ V.T + rho * np.outer(s, s)
        
        x = x_new
        x_history.append(x.copy())
        f_history.append(f(x))
    
    return np.array(x_history), np.array(f_history)

# Test on 2D quadratic
kappa = 100
A = np.diag([1.0, kappa])
def f_q(x): return 0.5 * x @ A @ x
def grad_f_q(x): return A @ x

x0 = np.array([1.0, 1.0])
x_hist, f_hist = bfgs(f_q, grad_f_q, x0, T=30)

print(f'BFGS on quadratic (kappa={kappa}):')
print(f'Converged in {len(f_hist)-1} iterations')
print(f'Final error: {f_hist[-1]:.2e}')

ok = f_hist[-1] < 1e-10
print(f"\n{'PASS' if ok else 'FAIL'} — BFGS converged")

In [ ]:
# === 4.2 BFGS vs GD vs Newton ===
if HAS_MPL:
    T = 50
    
    # GD
    x = x0.copy()
    err_gd = []
    for _ in range(T):
        x = x - (1/kappa) * (A @ x)
        err_gd.append(0.5 * x @ A @ x)
    
    # Newton (already have from section 2)
    err_newton = [0.0] * T  # Converges in 1 step
    
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    
    ax.plot(steps, err_gd, color=COLORS['neutral'], label='GD')
    ax.plot(steps, f_hist[1:], color=COLORS['primary'], label='BFGS')
    ax.plot(steps[:2], [f_q(x0), 0], 'o--', color=COLORS['error'], label='Newton')
    
    ax.set_yscale('log')
    ax.set_title('BFGS vs GD vs Newton on ill-conditioned quadratic')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Suboptimality')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 5. L-BFGS: Limited-Memory BFGS

Demonstrate the two-loop recursion for memory-efficient quasi-Newton.

In [ ]:
# === 5.1 L-BFGS two-loop recursion ===
def lbfgs_two_loop(g, s_list, y_list, rho_list, gamma):
    q = g.copy()
    m = len(s_list)
    alpha = np.zeros(m)
    
    for i in range(m - 1, -1, -1):
        alpha[i] = rho_list[i] * np.dot(s_list[i], q)
        q = q - alpha[i] * y_list[i]
    
    r = gamma * q
    
    for i in range(m):
        beta = rho_list[i] * np.dot(y_list[i], r)
        r = r + (alpha[i] - beta) * s_list[i]
    
    return -r

def lbfgs(f, grad_f, x0, m=10, T=100, tol=1e-10):
    x = x0.copy().astype(float)
    s_list, y_list, rho_list = [], [], []
    x_history = [x.copy()]
    f_history = [f(x)]
    
    g = grad_f(x)
    for t in range(T):
        if np.linalg.norm(g) < tol:
            break
        
        gamma = 1.0
        if s_list:
            gamma = np.dot(s_list[-1], y_list[-1]) / np.dot(y_list[-1], y_list[-1])
        
        d = lbfgs_two_loop(g, s_list, y_list, rho_list, gamma)
        
        # Line search
        eta = 1.0
        f_x = f(x)
        slope = g @ d
        while f(x + eta*d) > f_x + 0.01*eta*slope:
            eta *= 0.5
        
        s = eta * d
        x_new = x + s
        g_new = grad_f(x_new)
        y = g_new - g
        
        ys = y @ s
        if ys > 1e-10:
            s_list.append(s)
            y_list.append(y)
            rho_list.append(1.0 / ys)
            if len(s_list) > m:
                s_list.pop(0)
                y_list.pop(0)
                rho_list.pop(0)
        
        x = x_new
        g = g_new
        x_history.append(x.copy())
        f_history.append(f(x))
    
    return np.array(x_history), np.array(f_history)

# Test with different memory sizes
for m in [3, 5, 10, 20]:
    x_hist, f_hist = lbfgs(f_q, grad_f_q, x0, m=m, T=50)
    print(f'L-BFGS (m={m:2d}): {len(f_hist)-1} iters, final error = {f_hist[-1]:.2e}')

ok = f_hist[-1] < 1e-10
print(f"\n{'PASS' if ok else 'FAIL'} — L-BFGS converged")

---

## 6. Gauss-Newton Method

Apply Gauss-Newton to nonlinear least squares.

In [ ]:
# === 6.1 Gauss-Newton for exponential fitting ===
# Fit y = a * exp(b * x) to data
x_data = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_data = np.array([1.1, 2.9, 8.1, 22.0, 59.5])

def residuals(params):
    a, b = params
    return y_data - a * np.exp(b * x_data)

def jacobian(params):
    a, b = params
    J = np.zeros((len(x_data), 2))
    J[:, 0] = -np.exp(b * x_data)       # d r_i / d a
    J[:, 1] = -a * x_data * np.exp(b * x_data)  # d r_i / d b
    return J

def gn_objective(params):
    r = residuals(params)
    return 0.5 * np.dot(r, r)

# Gauss-Newton iterations
params = np.array([1.0, 1.0])  # Initial guess
T = 20
obj_hist = [gn_objective(params)]

for t in range(T):
    J = jacobian(params)
    r = residuals(params)
    
    # Solve (J^T J) d = -J^T r
    JtJ = J.T @ J
    Jtr = J.T @ r
    
    # Add regularization for stability
    JtJ += 1e-8 * np.eye(2)
    d = -la.solve(JtJ, Jtr)
    
    params = params + d
    obj_hist.append(gn_objective(params))
    
    if np.linalg.norm(d) < 1e-10:
        break

print(f'Gauss-Newton for y = a * exp(b * x)')
print(f'Fitted: a = {params[0]:.6f}, b = {params[1]:.6f}')
print(f'Final objective: {obj_hist[-1]:.2e}')
print(f'Iterations: {len(obj_hist)-1}')

ok = obj_hist[-1] < 1.0
print(f"\n{'PASS' if ok else 'FAIL'} — Gauss-Newton found a good fit")

In [ ]:
# === 6.2 Plot fit ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x_fit = np.linspace(-0.5, 4.5, 100)
    y_fit = params[0] * np.exp(params[1] * x_fit)
    
    axes[0].scatter(x_data, y_data, color=COLORS['primary'], s=60, label='Data')
    axes[0].plot(x_fit, y_fit, color=COLORS['error'], linewidth=2, label='Gauss-Newton fit')
    axes[0].set_title('Exponential curve fitting')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')
    axes[0].legend()
    
    axes[1].plot(range(len(obj_hist)), obj_hist, 'o-', color=COLORS['primary'])
    axes[1].set_yscale('log')
    axes[1].set_title('Gauss-Newton convergence')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Objective (log scale)')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 7. Natural Gradient Descent

Compare natural gradient with standard gradient for a Bernoulli model.

In [ ]:
# === 7.1 Natural gradient for Bernoulli ===
# Model: p(x|theta) = theta^x * (1-theta)^(1-x)
# Fisher: F(theta) = 1 / (theta * (1 - theta))
# Natural gradient: theta - x (much simpler than standard gradient)

np.random.seed(42)
true_theta = 0.7
data = np.random.binomial(1, true_theta, size=1000)

def neg_log_likelihood(theta, data):
    return -np.mean(data * np.log(theta + 1e-15) + (1-data) * np.log(1-theta + 1e-15))

    def standard_grad(theta, data):
    return -np.mean(data / (theta + 1e-15) - (1-data) / (1-theta + 1e-15))

def fisher(theta):
    return 1.0 / (theta * (1 - theta) + 1e-15)

def natural_grad(theta, data):
    g = standard_grad(theta, data)
    return g / fisher(theta)  # F^{-1} * g

# Compare SGD vs Natural GD
theta_sgd = 0.3
theta_ngd = 0.3
eta = 0.1
T = 100

err_sgd = []
err_ngd = []

for t in range(T):
    # Use full batch for fair comparison
    g_sgd = standard_grad(theta_sgd, data)
    g_ngd = natural_grad(theta_ngd, data)
    
    theta_sgd = np.clip(theta_sgd - eta * g_sgd, 0.01, 0.99)
    theta_ngd = np.clip(theta_ngd - eta * g_ngd, 0.01, 0.99)
    
    err_sgd.append(abs(theta_sgd - true_theta))
    err_ngd.append(abs(theta_ngd - true_theta))

err_sgd = np.array(err_sgd)
err_ngd = np.array(err_ngd)

print(f'True theta: {true_theta}')
print(f'SGD estimate: {theta_sgd:.6f}, error: {err_sgd[-1]:.6f}')
print(f'Natural GD estimate: {theta_ngd:.6f}, error: {err_ngd[-1]:.6f}')

ok = err_ngd[-1] <= err_sgd[-1]
print(f"\n{'PASS' if ok else 'FAIL'} — natural GD performs at least as well")

In [ ]:
# === 7.2 Plot comparison ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    
    ax.plot(steps, err_sgd, color=COLORS['neutral'], label='Standard GD')
    ax.plot(steps, err_ngd, color=COLORS['primary'], label='Natural GD')
    
    ax.set_yscale('log')
    ax.set_title('Natural GD vs Standard GD for Bernoulli estimation')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('|theta - theta_true|')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 8. Levenberg-Marquardt Method

Damped Gauss-Newton with adaptive regularization.

In [ ]:
# === 8.1 Levenberg-Marquardt ===
def levenberg_marquardt(residuals, jacobian, params0, T=50, mu=1e-3, nu=10):
    params = params0.copy().astype(float)
    obj_hist = [0.5 * np.dot(residuals(params), residuals(params))]
    mu_hist = [mu]
    
    for t in range(T):
        J = jacobian(params)
        r = residuals(params)
        JtJ = J.T @ J
        Jtr = J.T @ r
        
        # Try step
        d = -la.solve(JtJ + mu * np.eye(len(params)), Jtr)
        new_params = params + d
        new_obj = 0.5 * np.dot(residuals(new_params), residuals(new_params))
        old_obj = obj_hist[-1]
        
        if new_obj < old_obj:
            params = new_params
            obj_hist.append(new_obj)
            mu = mu / nu  # Trust the model more
        else:
            obj_hist.append(old_obj)
            mu = mu * nu  # Trust the model less
        
        mu_hist.append(mu)
        
        if np.linalg.norm(d) < 1e-10:
            break
    
    return params, np.array(obj_hist), np.array(mu_hist)

params0 = np.array([0.5, 0.5])
params_lm, obj_lm, mu_lm = levenberg_marquardt(residuals, jacobian, params0)

print(f'Levenberg-Marquardt result:')
print(f'a = {params_lm[0]:.6f}, b = {params_lm[1]:.6f}')
print(f'Final objective: {obj_lm[-1]:.2e}')
print(f'Final mu: {mu_lm[-1]:.2e}')
print(f'Iterations: {len(obj_lm)-1}')

ok = obj_lm[-1] < 1.0
print(f"\n{'PASS' if ok else 'FAIL'} — LM converged")

---

## 9. Hessian Spectrum Analysis

Visualize the eigenvalue distribution of the Hessian for a neural network.

In [ ]:
# === 9.1 Hessian eigenvalues for a small neural network ===
np.random.seed(42)
n_data, d_in, d_hidden, d_out = 100, 5, 10, 1
X = np.random.randn(n_data, d_in)
y = np.sin(X[:, 0:1]).sum(axis=1) + 0.1 * np.random.randn(n_data)

# Flatten parameters
W1 = np.random.randn(d_in, d_hidden) * 0.5
b1 = np.zeros(d_hidden)
W2 = np.random.randn(d_hidden, d_out) * 0.5
b2 = np.zeros(d_out)

def get_params():
    return np.concatenate([W1.ravel(), b1, W2.ravel(), b2])

    def set_params(p):
    global W1, b1, W2, b2
    W1 = p[:d_in*d_hidden].reshape(d_in, d_hidden)
    b1 = p[d_in*d_hidden:d_in*d_hidden+d_hidden]
    W2 = p[d_in*d_hidden+d_hidden:d_in*d_hidden+d_hidden+d_hidden*d_out].reshape(d_hidden, d_out)
    b2 = p[d_in*d_hidden+d_hidden+d_hidden*d_out:]

def relu(z): return np.maximum(0, z)

def forward(X):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    return a1 @ W2 + b2

def loss_fn(p):
    set_params(p)
    pred = forward(X).flatten()
    return 0.5 * np.mean((pred - y)**2)

p0 = get_params()
n_params = len(p0)
print(f'Number of parameters: {n_params}')

# Compute Hessian using finite differences (small network only)
eps = 1e-5
g0 = opt.approx_fprime(p0, loss_fn, eps)
H = np.zeros((n_params, n_params))
for i in range(n_params):
    p_plus = p0.copy()
    p_plus[i] += eps
    g_plus = opt.approx_fprime(p_plus, loss_fn, eps)
    H[:, i] = (g_plus - g0) / eps
H = 0.5 * (H + H.T)  # Symmetrize

eigvals = la.eigvalsh(H)
eigvals = np.sort(eigvals)[::-1]  # Descending

print(f'\nHessian eigenvalue spectrum:')
print(f'  Max eigenvalue: {eigvals[0]:.4f}')
print(f'  Min eigenvalue: {eigvals[-1]:.4f}')
print(f'  Condition number: {abs(eigvals[0]/eigvals[-1]):.1f}')
print(f'  Negative eigenvalues: {np.sum(eigvals < 0)}')
print(f'  Near-zero eigenvalues (|λ| < 0.01): {np.sum(np.abs(eigvals) < 0.01)}')

In [ ]:
# === 9.2 Plot eigenvalue spectrum ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Eigenvalue distribution
    axes[0].plot(range(len(eigvals)), eigvals, '.', color=COLORS['primary'], markersize=4)
    axes[0].axhline(0, color=COLORS['neutral'], linestyle='--')
    axes[0].set_title('Hessian eigenvalue spectrum')
    axes[0].set_xlabel('Eigenvalue index (sorted)')
    axes[0].set_ylabel('Eigenvalue')
    
    # Histogram
    axes[1].hist(eigvals, bins=30, color=COLORS['primary'], alpha=0.7, edgecolor='white')
    axes[1].axvline(0, color=COLORS['error'], linestyle='--')
    axes[1].set_title('Eigenvalue distribution')
    axes[1].set_xlabel('Eigenvalue')
    axes[1].set_ylabel('Count')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## Summary

This notebook demonstrated:

1. **Newton's method** — quadratic convergence on a non-quadratic function
2. **Newton vs GD** — Newton converges in one step for quadratics
3. **Damped Newton** — global convergence via backtracking on Rosenbrock
4. **BFGS** — superlinear convergence via quasi-Newton updates
5. **L-BFGS** — memory-efficient BFGS with two-loop recursion
6. **Gauss-Newton** — nonlinear least squares fitting
7. **Natural gradient** — reparameterization-invariant optimization
8. **Levenberg-Marquardt** — adaptive damping for robust convergence
9. **Hessian spectrum** — eigenvalue analysis of a neural network loss

See [exercises.ipynb](exercises.ipynb) for graded problems on these topics.

---

## 10. K-FAC: Kronecker-Factored Approximate Curvature

Demonstrate the K-FAC approximation for a single linear layer.

In [ ]:
# === 10.1 K-FAC for a linear layer ===
# For a layer Z = XW + b, the Fisher block is approximated as:
# F ≈ A ⊗ G where A = E[aa^T] (input covariance) and G = E[gg^T] (output gradient covariance)

np.random.seed(42)
n_batch, d_in, d_out = 1000, 20, 10

# Simulate input activations and output gradients
A_data = np.random.randn(n_batch, d_in)
G_data = np.random.randn(n_batch, d_out)

# Empirical Fisher (full, for comparison)
# F_full ≈ (1/n) sum_i vec(a_i g_i^T) vec(a_i g_i^T)^T
# This is (d_in * d_out)^2 = 40000 entries — too large for direct computation
# Instead, we verify the Kronecker structure

A_cov = A_data.T @ A_data / n_batch  # d_in x d_in
G_cov = G_data.T @ G_data / n_batch  # d_out x d_out

print(f'K-FAC for linear layer: d_in={d_in}, d_out={d_out}')
print(f'Input covariance shape: {A_cov.shape}')
print(f'Output gradient covariance shape: {G_cov.shape}')
print(f'Full Fisher would be: {(d_in*d_out)**2} entries')
print(f'K-FAC storage: {d_in**2 + d_out**2} entries')
print(f'Compression ratio: {(d_in*d_out)**2 / (d_in**2 + d_out**2):.1f}x')

# Verify Kronecker product structure
# The Fisher for a linear layer with MSE loss is approximately A_cov ⊗ G_cov
F_kfac = np.kron(G_cov, A_cov)  # Note: order is G ⊗ A for vec(W)
print(f'\nK-FAC Fisher shape: {F_kfac.shape}')
print(f'K-FAC Fisher is PSD: {np.all(la.eigvalsh(F_kfac) >= -1e-10)}')

# Compare with a small-scale exact Fisher
d_in_s, d_out_s, n_s = 5, 3, 500
A_s = np.random.randn(n_s, d_in_s)
G_s = np.random.randn(n_s, d_out_s)

# Exact Fisher: F_ij,kl = E[a_i a_k g_j g_l]
F_exact = np.zeros((d_in_s*d_out_s, d_in_s*d_out_s))
for i in range(n_s):
    vec_i = np.outer(A_s[i], G_s[i]).ravel()  # vec(a g^T)
    F_exact += np.outer(vec_i, vec_i)
F_exact /= n_s

A_cov_s = A_s.T @ A_s / n_s
G_cov_s = G_s.T @ G_s / n_s
F_kfac_s = np.kron(G_cov_s, A_cov_s)

# Measure approximation quality
rel_error = la.norm(F_exact - F_kfac_s) / la.norm(F_exact)
print(f'\nSmall-scale verification:')
print(f'Exact Fisher shape: {F_exact.shape}')
print(f'Relative error ||F_exact - F_KFAC|| / ||F_exact||: {rel_error:.4f}')

ok = rel_error < 0.5  # K-FAC is an approximation, not exact
print(f"\n{'PASS' if ok else 'FAIL'} — K-FAC approximates the Fisher")

---

## 11. Conjugate Gradient for Newton Systems

Solve the Newton system Hx = -g using conjugate gradient without forming H^{-1}.

In [ ]:
# === 11.1 CG solver ===
def conjugate_gradient(H_func, b, x0=None, max_iter=None, tol=1e-10):
    """Solve Hx = b using conjugate gradient.
    H_func: function that computes H @ v
    """
    n = len(b)
    if max_iter is None:
        max_iter = n
    
    x = np.zeros(n) if x0 is None else x0.copy()
    r = b - H_func(x)
    p = r.copy()
    rs_old = r @ r
    
    residuals = [np.sqrt(rs_old)]
    
    for i in range(max_iter):
        Hp = H_func(p)
        alpha = rs_old / (p @ Hp)
        x = x + alpha * p
        r = r - alpha * Hp
        rs_new = r @ r
        residuals.append(np.sqrt(rs_new))
        
        if np.sqrt(rs_new) < tol:
            break
        
        beta = rs_new / rs_old
        p = r + beta * p
        rs_old = rs_new
    
    return x, np.array(residuals)

# Test on a known system
n = 100
A = np.random.randn(n, n)
H = A.T @ A + np.eye(n)  # SPD matrix
b = np.random.randn(n)

x_cg, residuals = conjugate_gradient(lambda v: H @ v, b, tol=1e-12)
x_exact = la.solve(H, b)

print(f'CG for {n}×{n} SPD system:')
print(f'CG iterations: {len(residuals)-1}')
print(f'||x_CG - x_exact||: {la.norm(x_cg - x_exact):.2e}')
print(f'||H x_CG - b||: {la.norm(H @ x_cg - b):.2e}')

ok = la.norm(x_cg - x_exact) < 1e-8
print(f"\n{'PASS' if ok else 'FAIL'} — CG solved the system accurately")

In [ ]:
# === 11.2 Plot CG convergence ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(len(residuals)), residuals, 'o-', color=COLORS['primary'])
    ax.set_yscale('log')
    ax.set_title(f'Conjugate Gradient convergence ({n}×{n} system)')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Residual norm ||r_k||')
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 12. Trust Region Method

Implement a trust region method as an alternative to line search.

In [ ]:
# === 12.1 Trust region for Rosenbrock ===
def trust_region(f, grad_f, hess_f, x0, Delta0=1.0, T=100, eta_thresh=0.1):
    x = x0.copy().astype(float)
    Delta = Delta0
    x_history = [x.copy()]
    f_history = [f(x)]
    Delta_history = [Delta]
    
    for t in range(T):
        g = grad_f(x)
        H = hess_f(x)
        
        # Regularize Hessian
        eigvals = la.eigvalsh(H)
        if eigvals[0] < 0:
            H = H + (-eigvals[0] + 1e-6) * np.eye(len(x))
        
        # Solve trust region subproblem (simplified: use dogleg)
        # Newton step
        d_newton = -la.solve(H, g)
        
        if np.linalg.norm(d_newton) <= Delta:
            d = d_newton
        else:
            # Scale Newton step to trust region boundary
            d = Delta * d_newton / np.linalg.norm(d_newton)
        
        # Compute actual and predicted reduction
        actual_red = f(x) - f(x + d)
        pred_red = -(g @ d + 0.5 * d @ H @ d)
        
        if pred_red > 1e-15:
            rho = actual_red / pred_red
        else:
            rho = 0.0
        
        if rho > 0.75 and np.linalg.norm(d) >= 0.9 * Delta:
            Delta = min(2 * Delta, 100.0)
        elif rho < 0.25:
            Delta = 0.25 * Delta
        
        if rho > eta_thresh:
            x = x + d
        
        x_history.append(x.copy())
        f_history.append(f(x))
        Delta_history.append(Delta)
        
        if np.linalg.norm(g) < 1e-10:
            break
    
    return np.array(x_history), np.array(f_history), np.array(Delta_history)

x0 = np.array([0.0, 0.0])
x_tr, f_tr, Delta_tr = trust_region(rosenbrock, rosenbrock_grad, rosenbrock_hess, x0)

print(f'Trust region on Rosenbrock:')
print(f'Start: {x0}, f = {f_tr[0]:.4f}')
print(f'Final: ({x_tr[-1][0]:.6f}, {x_tr[-1][1]:.6f}), f = {f_tr[-1]:.2e}')
print(f'Iterations: {len(f_tr)-1}')
print(f'Final trust radius: {Delta_tr[-1]:.4f}')

ok = f_tr[-1] < 1e-6
print(f"\n{'PASS' if ok else 'FAIL'} — trust region converged")

---

## 13. Hessian-Vector Products via Pearlmutter's Trick

Compute H @ v without forming the full Hessian.

In [ ]:
# === 13.1 HVP for a simple function ===
# For f(x) = x^T A x / 2, H = A, so H @ v = A @ v
# We verify this using finite differences

n = 50
A = np.random.randn(n, n)
A = A.T @ A + np.eye(n)  # Make SPD

def f_hvp(x):
    return 0.5 * x @ A @ x

def grad_f_hvp(x):
    return A @ x

def hvp_finite_diff(grad_fn, x, v, eps=1e-6):
    """Compute H @ v using finite differences of the gradient."""
    return (grad_fn(x + eps*v) - grad_fn(x - eps*v)) / (2*eps)

x = np.random.randn(n)
v = np.random.randn(n)

# Exact: H @ v = A @ v
Hv_exact = A @ v

# Finite difference approximation
Hv_fd = hvp_finite_diff(grad_f_hvp, x, v)

rel_error = la.norm(Hv_exact - Hv_fd) / la.norm(Hv_exact)

print(f'Hessian-vector product verification (n={n}):')
print(f'||Hv_exact - Hv_fd|| / ||Hv_exact||: {rel_error:.2e}')

ok = rel_error < 1e-4
print(f"\n{'PASS' if ok else 'FAIL'} — HVP via finite differences is accurate")

---

## 14. Newton's Method for Logistic Regression

Apply Newton's method (IRLS) to logistic regression.

In [ ]:
# === 14.1 IRLS for logistic regression ===
np.random.seed(42)
n_data, d_feat = 500, 10
X_lr = np.random.randn(n_data, d_feat)
w_true = np.array([1.0, -0.5, 0.8, -0.3, 0.6, 0.2, -0.4, 0.7, -0.1, 0.3])
logits = X_lr @ w_true
probs = 1.0 / (1.0 + np.exp(-logits))
y_lr = (probs > 0.5).astype(float)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def neg_log_likelihood(w, X, y):
    z = X @ w
    return -np.mean(y * np.log(sigmoid(z) + 1e-15) + (1-y) * np.log(1-sigmoid(z) + 1e-15))

def irls(X, y, T=20):
    n, d = X.shape
    w = np.zeros(d)
    obj_hist = [neg_log_likelihood(w, X, y)]
    
    for t in range(T):
        z = X @ w
        p = sigmoid(z)
        
        # Gradient: X^T (p - y) / n
        g = X.T @ (p - y) / n
        
        # Hessian: X^T S X / n where S = diag(p * (1-p))
        S = p * (1 - p)
        H = (X * S[:, None]).T @ X / n
        
        # Regularize
        H += 1e-8 * np.eye(d)
        
        # Newton step
        d_w = -la.solve(H, g)
        w = w + d_w
        
        obj_hist.append(neg_log_likelihood(w, X, y))
        
        if np.linalg.norm(d_w) < 1e-10:
            break
    
    return w, np.array(obj_hist)

w_newton, obj_newton = irls(X_lr, y_lr, T=15)

print(f'Newton/IRLS for logistic regression:')
print(f'n={n_data}, d={d_feat}')
print(f'Converged in {len(obj_newton)-1} iterations')
print(f'Final NLL: {obj_newton[-1]:.6f}')
print(f'||w_newton - w_true||: {la.norm(w_newton - w_true):.6f}')

ok = obj_newton[-1] < 0.3
print(f"\n{'PASS' if ok else 'FAIL'} — IRLS converged")

In [ ]:
# === 14.2 Plot IRLS convergence ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(len(obj_newton)), obj_newton, 'o-', color=COLORS['primary'])
    ax.set_yscale('log')
    ax.set_title('IRLS (Newton) convergence for logistic regression')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Negative log-likelihood')
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 15. Summary: All Methods Compared

Final comparison of second-order methods on a standard test problem.

In [ ]:
# === 15.1 Comprehensive comparison ===
def run_comparison():
    # Test problem: Rosenbrock
    x0 = np.array([0.0, 0.0])
    target = 1e-8
    
    results = {}
    
    # 1. Damped Newton
    x_hist, f_hist, _ = damped_newton(rosenbrock, rosenbrock_grad, rosenbrock_hess, x0, T=100)
    for i, f_val in enumerate(f_hist):
        if f_val < target:
            results['Damped Newton'] = i
            break
    
    # 2. BFGS
    x_hist, f_hist = bfgs(rosenbrock, rosenbrock_grad, x0, T=200)
    for i, f_val in enumerate(f_hist):
        if f_val < target:
            results['BFGS'] = i
            break
    
    # 3. L-BFGS
    x_hist, f_hist = lbfgs(rosenbrock, rosenbrock_grad, x0, m=10, T=200)
    for i, f_val in enumerate(f_hist):
        if f_val < target:
            results['L-BFGS (m=10)'] = i
            break
    
    # 4. Trust region
    x_hist, f_hist, _ = trust_region(rosenbrock, rosenbrock_grad, rosenbrock_hess, x0, T=100)
    for i, f_val in enumerate(f_hist):
        if f_val < target:
            results['Trust Region'] = i
            break
    
    # 5. scipy L-BFGS-B
    res = opt.minimize(rosenbrock, x0, method='L-BFGS-B',
                       jac=rosenbrock_grad, options={'ftol': target})
    results['scipy L-BFGS-B'] = res.nit
    
    return results

results = run_comparison()

print('Iterations to reach f < 1e-8 on Rosenbrock:')
print('-' * 40)
for name, iters in sorted(results.items(), key=lambda x: x[1]):
    print(f'{name:<25} | {iters}')

if results:
    fastest = min(results.values())
    slowest = max(results.values())
    print(f'\nSpeedup (slowest/fastest): {slowest/fastest:.1f}x')

---

## 16. Fisher Information for Common Distributions

Compute and visualize the Fisher information matrix for several distributions.

In [ ]:
# === 16.1 Fisher for Gaussian ===
# N(mu, sigma^2): F = diag(1/sigma^2, 1/(2*sigma^4))

sigma_vals = np.linspace(0.5, 3.0, 50)
F_mu = 1.0 / sigma_vals**2
F_sigma = 1.0 / (2 * sigma_vals**4)

print('Fisher information for N(mu, sigma^2):')
print('F_mu = 1/sigma^2, F_sigma = 1/(2*sigma^4)')
print(f'At sigma=1: F_mu={F_mu[sigma_vals==1.0][0]:.4f}, F_sigma={F_sigma[sigma_vals==1.0][0]:.4f}')
print(f'At sigma=2: F_mu={F_mu[sigma_vals==2.0][0]:.4f}, F_sigma={F_sigma[sigma_vals==2.0][0]:.4f}')

print('\nInterpretation:')
print('  Larger sigma -> smaller Fisher -> less information per sample')
print('  F_mu decreases as 1/sigma^2 (slowly)')
print('  F_sigma decreases as 1/sigma^4 (rapidly)')

In [ ]:
# === 16.2 Plot Fisher vs sigma ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(sigma_vals, F_mu, color=COLORS['primary'], label='F_mu = 1/sigma^2')
    ax.plot(sigma_vals, F_sigma, color=COLORS['secondary'], label='F_sigma = 1/(2*sigma^4)')
    ax.set_title('Fisher information for Gaussian distribution')
    ax.set_xlabel('sigma')
    ax.set_ylabel('Fisher information')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 17. BFGS Superlinear Convergence

Verify superlinear convergence of BFGS on a non-quadratic function.

In [ ]:
# === 17.1 BFGS on non-quadratic ===
def rosenbrock_4d(x):
    """4D Rosenbrock."""
    return sum(100*(x[i+1]-x[i]**2)**2 + (1-x[i])**2 for i in range(len(x)-1))

def rosenbrock_4d_grad(x):
    n = len(x)
    g = np.zeros(n)
    for i in range(n-1):
        g[i] += -400*x[i]*(x[i+1]-x[i]**2) - 2*(1-x[i])
        g[i+1] += 200*(x[i+1]-x[i]**2)
    return g

x0 = np.array([0.0, 0.0, 0.0, 0.0])
x_hist, f_hist = bfgs(rosenbrock_4d, rosenbrock_4d_grad, x0, T=100)

print(f'BFGS on 4D Rosenbrock:')
print(f'Iterations: {len(f_hist)-1}')
print(f'Final value: {f_hist[-1]:.2e}')
print(f'Final point: {x_hist[-1]}')

# Check superlinear convergence: ratio of successive errors should go to 0
f_star = 0.0
errors = f_hist[1:] - f_star
errors = errors[errors > 1e-15]
if len(errors) >= 3:
    ratios = errors[1:] / errors[:-1]
    print(f'\nConvergence ratios (should decrease for superlinear):')
    for i, r in enumerate(ratios[-5:]):
        print(f'  ratio[{i}] = {r:.6f}')

ok = f_hist[-1] < 1e-10
print(f"\n{'PASS' if ok else 'FAIL'} — BFGS converged superlinearly")

In [ ]:
# === 17.2 Plot BFGS convergence ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, len(f_hist))
    ax.plot(steps, f_hist[1:], 'o-', color=COLORS['primary'])
    ax.set_yscale('log')
    ax.set_title('BFGS superlinear convergence on 4D Rosenbrock')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('f(x) (log scale)')
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 18. Affine Invariance of Newton's Method

Demonstrate that Newton's method is invariant under linear reparameterization.

In [ ]:
# === 18.1 Affine invariance test ===
# f(x) = 0.5 * x^T A x
# Reparameterize: y = B x, so x = B^{-1} y
# g(y) = f(B^{-1} y) = 0.5 * y^T (B^{-T} A B^{-1}) y

A = np.array([[2.0, 1.0], [1.0, 3.0]])
B = np.array([[2.0, 1.0], [0.5, 1.5]])  # Non-singular transformation
A_new = np.linalg.inv(B).T @ A @ np.linalg.inv(B)

x0 = np.array([1.0, 2.0])
y0 = B @ x0

# Newton in x-space
x = x0.copy()
x_path = [x.copy()]
for _ in range(5):
    x = x - np.linalg.solve(A, A @ x)
    x_path.append(x.copy())

# Newton in y-space
y = y0.copy()
y_path = [y.copy()]
for _ in range(5):
    y = y - np.linalg.solve(A_new, A_new @ y)
    y_path.append(y.copy())

# Check: y_t should equal B @ x_t
print('Affine invariance test:')
for t in range(len(x_path)):
    y_expected = B @ x_path[t]
    y_actual = y_path[t]
    diff = np.linalg.norm(y_expected - y_actual)
    print(f'  t={t}: ||B@x_t - y_t|| = {diff:.2e}')

max_diff = max(np.linalg.norm(B @ x_path[t] - y_path[t]) for t in range(len(x_path)))
ok = max_diff < 1e-10
print(f"\n{'PASS' if ok else 'FAIL'} — Newton's method is affine invariant")

---

## 19. scipy.optimize Second-Order Methods

Compare different scipy.optimize methods on a test problem.

In [ ]:
# === 19.1 scipy.optimize comparison ===
from scipy.optimize import minimize

x0 = np.array([0.0, 0.0])

methods = ['Newton-CG', 'BFGS', 'L-BFGS-B', 'trust-ncg', 'trust-krylov']
results = {}

for method in methods:
    try:
        res = minimize(rosenbrock, x0, method=method,
                       jac=rosenbrock_grad, hess=rosenbrock_hess,
                       options={'maxiter': 200, 'ftol': 1e-12})
        results[method] = (res.nit, res.fun, res.success)
    except Exception as e:
        results[method] = (None, None, False)
        print(f'{method}: failed ({e})')

print(f'scipy.optimize on Rosenbrock from {x0}:')
print(f'{"Method":<20} | {"Iters":>5} | {"f(x)":>12} | Success')
print('-' * 50)
for method, (nit, fun, success) in results.items():
    nit_str = str(nit) if nit is not None else 'N/A'
    fun_str = f'{fun:.2e}' if fun is not None else 'N/A'
    print(f'{method:<20} | {nit_str:>5} | {fun_str:>12} | {success}')

ok = any(success for _, (_, _, success) in results.items())
print(f"\n{'PASS' if ok else 'FAIL'} — at least one method succeeded")